# 05 천장 확인: 확장 피처 150개 × (L1 로지스틱 회귀, RandomForest, ExtraTrees, HistGradientBoosting)

- 목적: 후보 피처를 한꺼번에 넣었을 때 F₂가 어디까지 오르는지 확인한다 (EXP-002)
- 평가: `src/evaluate.py`의 중첩 CV (EXP-001과 같은 분할·seed·임계값 규칙)
- 기준선: core 13개 × 로지스틱 회귀 = EXP-001의 0.608이 다시 나와야 비교가 성립한다
- 판정: ≥ 0.90 하나씩 빼며 설명 / 0.70–0.90 도메인 개선 방법 / < 0.70 원신호 직접 분석

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_dataset
from src.features import feature_table_ext
from src.evaluate import nested_cv

X, y, names = load_dataset(ROOT / "data" / "TeamProject_train_변경")
E = feature_table_ext(X)                 # 150 × 150 (앞 13열이 core)
CORE = list(E.columns[:13])
F_core = E[CORE].to_numpy()
F_ext = E.to_numpy()
groups = np.arange(len(y))               # 파일 하나 = 그룹 하나
print("core", F_core.shape, "확장", F_ext.shape)

# (피처 세트, 모델 생성 함수). 호출할 때마다 새 모델 → fold마다 처음부터 학습
RUNS = {
    "core13 · 로지스틱(EXP-001 재현)": (F_core, lambda: make_pipeline(
        RobustScaler(), LogisticRegression(class_weight="balanced", max_iter=5000))),
    "확장150 · L1 로지스틱": (F_ext, lambda: make_pipeline(
        RobustScaler(), LogisticRegression(l1_ratio=1, solver="liblinear", class_weight="balanced"))),
    "확장150 · RandomForest": (F_ext, lambda: RandomForestClassifier(
        n_estimators=500, class_weight="balanced_subsample", min_samples_leaf=2, n_jobs=-1, random_state=0)),
    "확장150 · ExtraTrees": (F_ext, lambda: ExtraTreesClassifier(
        n_estimators=500, class_weight="balanced_subsample", min_samples_leaf=2, n_jobs=-1, random_state=0)),
    "확장150 · HistGB": (F_ext, lambda: HistGradientBoostingClassifier(
        class_weight="balanced", max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}

## 1. 정직한 점수 (중첩 CV, seed 5개)

In [ ]:
results = {name: nested_cv(F, y, groups, make) for name, (F, make) in RUNS.items()}

rows = []
for name, r in results.items():
    rows.append({
        "실행": name,
        "F2 평균": r["f2"].mean(), "F2 표준편차": r["f2"].std(),
        "재현율": r["recall"].mean(), "정밀도": r["precision"].mean(),
        "놓친 고장(평균)": r["miss_counts"].sum() / len(r["f2"]),
        "오경보(평균)": r["fp_counts"].sum() / len(r["f2"]),
    })
summary = pd.DataFrame(rows)
print(summary.round(3).to_string(index=False))
for name, r in results.items():
    print(f"{name} seed별 F2:", r["f2"].round(3))

## 2. EXP-001에서 놓친 고장 4개는 잡혔나

숫자 = seed 5개 중 놓친 횟수 (0이면 매번 잡음)

In [ ]:
names_arr = np.array(names)
HARD = ["abn_002", "abn_022", "abn_023", "abn_029"]
idx = [names.index(n) for n in HARD]
print(pd.DataFrame({name: r["miss_counts"][idx] for name, r in results.items()}, index=HARD).T.to_string())
for name, r in results.items():
    print(f"\n{name}")
    print("  자주 놓친 고장(3회 이상):", names_arr[r["miss_counts"] >= 3].tolist())
    print("  자주 오경보 정상(3회 이상) 개수:", int((r["fp_counts"] >= 3).sum()))

## 3. 어떤 피처를 썼나 (해석용, 점수 아님)

150개 전체로 RandomForest를 한 번 학습해 불순도 중요도를 본다. 전체 데이터를 썼으므로 점수 계산에는 쓰지 않는다.
피처 묶음(id 앞 3글자)별 합계도 함께 본다.

In [ ]:
rf = RUNS["확장150 · RandomForest"][1]().fit(F_ext, y)
imp = pd.Series(rf.feature_importances_, index=E.columns).sort_values(ascending=False)
print(imp.head(20).round(4).to_string())
print("\n묶음별 중요도 합:")
print(imp.groupby(imp.index.str[:3]).sum().sort_values(ascending=False).round(3).head(12).to_string())